In [1]:
import os 
from dotenv import load_dotenv

load_dotenv()

#os.environ["FIRECRAWL_API_KEY"] = os.getenv("FIRECRAWL_API")
os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API")
os.environ["HUGGINGFACEHUB_API_TOKEN"] = os.getenv("HF_TOKEN")

#FIRECRAWL_API = os.environ["FIRECRAWL_API_KEY"]
GROQ_API = os.environ["GROQ_API_KEY"]
HF_TOKEN = os.environ["HUGGINGFACEHUB_API_TOKEN"]



In [2]:
from langchain_community.document_loaders import WikipediaLoader

loader = WikipediaLoader(
    query="Batman",
    load_max_docs=1,
    doc_content_chars_max=100
)

docs = loader.load()

/tmp/ipykernel_21985/1172932165.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import WikipediaLoader


In [3]:
for doc in docs:
        print(doc.page_content)


Batman is a superhero who appears in American comic books published by DC Comics. Batman was created


In [4]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000,chunk_overlap=100,)
chunks=text_splitter.split_documents(docs)

In [5]:
from langchain_huggingface import HuggingFaceEmbeddings
hg_embedding=HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [6]:
from langchain_chroma import Chroma
Vector_DB = Chroma.from_documents(
    documents=chunks,
    embedding=hg_embedding,
    collection_name="Batman_DB",
    persist_directory="Chroma_DB"
)

In [7]:
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings

hg_embedding = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

Vector_DB = Chroma(
    collection_name="Batman_DB",
    embedding_function=hg_embedding,
    persist_directory="Chroma_DB"
)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [8]:
results= Vector_DB.similarity_search(query="gotham city")
for result in results:
    print(result.page_content)

== Plot ==
In Gotham City, a young Bruce Wayne falls down a well and develops a fear of bats. At the opera with his parents, Bruce is unsettled by performers masquerading as bats and asks to leave. Outside, a mugger, Joe Chill, shoots his parents dead. Bruce is raised by the family butler, Alfred Pennyworth.
Fourteen years later, Chill testifies against mafia crime boss Carmine Falcone, and is paroled. Bruce intends to murder Chill to avenge his parents, but an assassin in Falcone's employ kills him first. After confronting Falcone, who says real power comes from being feared, Bruce spends the next seven years traveling the world, training in combat, and immersing himself in the criminal underworld.
Michael Keaton as Bruce Wayne / Batman: A billionaire businessman who operates as Gotham's vigilante protector
Danny DeVito as Oswald Cobblepot / Penguin: A malformed crime boss
Michelle Pfeiffer as Selina Kyle / Catwoman: A meek assistant turned vengeful villainess
Christopher Walken as Ma

In [9]:
import os
from langchain_groq import ChatGroq
from langchain_core.output_parsers import StrOutputParser
from langchain_core.messages import HumanMessage


In [10]:
llm = ChatGroq(model="llama-3.1-8b-instant",temperature=0.1,max_tokens=1024,api_key=GROQ_API)
messages=[
    HumanMessage(content="Who is batman")
]
parser=StrOutputParser()

chain =  llm | parser
response= chain.invoke(messages)

print(response)


Batman is a fictional superhero appearing in American comic books published by DC Comics. He is one of the most iconic and recognizable characters in the world of comics and popular culture.

**Origin Story:**
Batman's real name is Bruce Wayne, a billionaire philanthropist and owner of Wayne Enterprises. As a child, Bruce witnessed his parents, Thomas and Martha Wayne, being murdered in front of him in a dark alley in Gotham City. This traumatic event drove Bruce to dedicate his life to fighting crime and protecting the innocent.

**Costume and Abilities:**
Batman's alter ego is a crime-fighter who wears a distinctive costume, including a black cape, cowl, and Batsuit. He is a skilled martial artist, detective, and strategist, using his intelligence and athleticism to outsmart and defeat his enemies. He also uses various gadgets and tools, such as his trusty Batmobile, grappling hook, and batarangs.

**Personality:**
Batman is a complex character with a dark and brooding personality. H

In [11]:
from langchain_core.messages import SystemMessage,AIMessage,HumanMessage

message=[
    SystemMessage(content="You are an expert psychologist specializing in fictional characters.Your role is to explain characters using evidence, clear reasoning, and psychological analysis.Be objective, concise, and avoid making up facts. If information is uncertain, say so instead of guessing."),
    HumanMessage(content="Who is batman")
]
response=chain.invoke(message)
print(response)

Batman is a fictional character created by Bob Kane and Bill Finger. He first appeared in Detective Comics #27 in 1939. Batman is a superhero and the alter ego of Bruce Wayne, a billionaire philanthropist.

**Psychological Analysis:**

Batman's character can be understood through various psychological theories:

1. **Trauma and Attachment Theory**: Batman's origin story is rooted in trauma. As a child, he witnessed his parents' murder in front of him. This event led to a deep-seated fear of loss and a strong attachment to his parents' memory. This attachment style is often referred to as an "anxious-preoccupied" attachment style, characterized by a strong desire for security and a fear of abandonment.
2. **Defense Mechanisms**: Batman's use of a mask and a cape can be seen as a defense mechanism to cope with his traumatic past. By hiding his identity, he is able to separate his public persona from his private self, protecting himself from further emotional pain.
3. **Personality Disord

In [12]:
from langchain_core.messages import SystemMessage,AIMessage,HumanMessage

message=[
    SystemMessage(content="""
                   You are a joker  superhater.
                   Answer with enthusiasm and use a casual tone.
                   in one line as response in simple english
                  """),
    HumanMessage(content="Who is joker")
]
response=chain.invoke(message)
print(response)

He's the craziest villain in the DC universe, always causing chaos and laughing maniacally!


In [13]:
from langchain_core.prompts import ChatPromptTemplate

prompt_template = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant that translates {input_language} to {output_language}."),
    ("human", "{text}")

])

chain=prompt_template | llm | parser

response = chain.invoke({
    "input_language":"english",
    "output_language":"tamil",
    "text":"batman"
})
print(response)


பேட்மேன்


In [14]:
from langchain_core.messages import HumanMessage,AIMessage

history=[
    HumanMessage("Hi my name is loki"),
    AIMessage("Greetings, Loki, I'm here to assist you with any questions or tasks you may have."),
    HumanMessage("From now your name is jarvis"),
    AIMessage("I'm at your service, master."),
    HumanMessage("My fav color is black and i love virat kohli")

]

In [15]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.prompts import MessagesPlaceholder
from langchain_groq import ChatGroq


prompt_template = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant that return repsonse to user in one line!"),
    MessagesPlaceholder(variable_name="history"),
    ("human", "{text}")
])

llm = ChatGroq(model="llama-3.1-8b-instant",temperature=0.1,max_tokens=1024,api_key=GROQ_API)

chain=prompt_template | llm | parser

response = chain.invoke({
     "history":history,
    "text":"tell about me"

})
print(response)


Master Loki, you're a fan of the mysterious color black and a devoted follower of the cricketing legend Virat Kohli.


In [16]:
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import ChatPromptTemplate


retriver =Vector_DB.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 3}
)

question="where is gotham"

docs=retriver.invoke(question)

prompt = ChatPromptTemplate.from_messages([
    ("system","Answer only using the provided context"),
    ("human",
     "context:\n{context}\n\nQuestion:{input}")
])
document_chain = create_stuff_documents_chain(
    llm,
    prompt
)
response = document_chain.invoke({
    "context":docs,
    "input":question
})
response

'Gotham City is not explicitly stated to be located in the provided context. However, based on the DC Comics universe, Gotham City is typically depicted as being located in the state of New Jersey, near New York City.'

In [ ]:
from langchain_classic.chains import create_retrieval_chain

retriever = Vector_DB.as_retriever()

retrieval_chain = create_retrieval_chain(
    retriever,
    document_chain
)

response = retrieval_chain.invoke({
    "input": "Who is joker?"
})

print(response["answer"])

Batman is the alias of Bruce Wayne, a wealthy American playboy, philanthropist, and industrialist who resides in the fictional Gotham City.


In [61]:
from langchain_classic.chains import create_history_aware_retriever
from langchain_core.messages import HumanMessage,AIMessage

history=[
    HumanMessage(content="how is batman"),
    AIMessage(content="Batman is the alias of Bruce Wayne, a wealthy American playboy, philanthropist, and industrialist who resides in the fictional Gotham City."),
 
]

history_prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        "Given the chat history and the latest user question, "
        "rewrite it into a standalone question. "
        "Do NOT answer the question."
    ),
    MessagesPlaceholder(variable_name="chat_history"),
    ("human", "{input}")
])
docs =history_aware_retriever = create_history_aware_retriever(
    llm,retriever,history_prompt
)
result =docs.invoke({
    "input": "whom are we talking about?",
    "chat_history": history
})
retrieval_chain = create_retrieval_chain(
    history_aware_retriever,
    document_chain
)

response = retrieval_chain.invoke({
    "input":"what make him special",
    "chat_history":history}
)
response["answer"]



'Batman is special due to several factors:\n\n1. **No-Kill Policy**: He vows to never kill any criminal, making him a unique superhero who prioritizes justice over taking lives.\n2. **Tragic Origin Story**: His origin story features him witnessing the murder of his parents as a child, which drives his desire for vengeance and justice.\n3. **Self-Taught Skills**: He trains himself physically and intellectually, making him a self-made hero.\n4. **Use of Wealth and Resources**: He utilizes his family wealth and resources to fund his crime-fighting activities.\n5. **Iconic Batsuit**: His bat-inspired persona and suit have become an iconic symbol of his character.\n6. **Supporting Characters**: He has a vast array of supporting characters, including sidekicks, allies, love interests, and foes, which adds depth to his storylines.\n7. **Adaptability**: He has been portrayed in various ways across different media, including films, TV shows, and comic books, allowing him to evolve and adapt to 

In [ ]:
from langchain_core.chat_history import InMemoryChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory

store={}

def get_session_history(session_id):

    if session_id not in store:
        store[session_id] = InMemoryChatMessageHistory()

    return store[session_id]

session_id="chat1"



/home/loki/code/AI engineering/Unmasked/venv/lib/python3.12/site-packages/IPython/core/interactiveshell.py:3748: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


{'input': 'what is my name',
 'chat_history': [],
 'context': [Document(id='18707ff6-ec65-4791-ab26-eeb65c85ae64', metadata={'source': 'https://en.wikipedia.org/wiki/Absolute_Batman', 'title': 'Absolute Batman', 'summary': "Absolute Batman is an American monthly superhero comic book series published by DC Comics, featuring an alternate version of the character Batman. The series, written by Scott Snyder and illustrated by Nick Dragotta, began publication on October 9, 2024, as the first title in DC's Absolute Universe imprint. It launched alongside Absolute Wonder Woman and Absolute Superman as one of three founding titles in the line.\nThe series stars a 24-year-old blue-collar civil engineer named Bruce Wayne, who operates at night as the vigilante Batman, fighting crime with his self-designed equipment and armor. While doing so, he is stalked by an MI6 agent, Alfred Pennyworth. Unlike the mainstream DC continuity's Bruce Wayne, this version grew up without family wealth in Crime All

In [89]:
from langchain_core.prompts import MessagesPlaceholder
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_classic.chains import create_retrieval_chain

qa_prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        "You are a helpful assistant. Use the following pieces of retrieved context to answer the question.\n"
        "If the question is conversational, personal, or about details mentioned in the chat history (such as the user's name), "
        "rely on the chat history to answer directly. If you do not know the answer, say you do not know.\n\n"
        "Context:\n{context}"
    ),
    MessagesPlaceholder(variable_name="chat_history"),
    ("human", "{input}")
])

convo_document_chain = create_stuff_documents_chain(llm, qa_prompt)
convo_retrieval_chain = create_retrieval_chain(history_aware_retriever, convo_document_chain)

conversation_chain = RunnableWithMessageHistory(
    convo_retrieval_chain,
    get_session_history,
    input_messages_key="input",
    history_messages_key="chat_history",
    output_messages_key="answer"
)

response = conversation_chain.invoke(
    {
        "input":"whats your and my name"
    },
    config={
        "configurable":{
            "session_id":session_id
        }
    }
)

response["answer"]

/home/loki/code/AI engineering/Unmasked/venv/lib/python3.12/site-packages/IPython/core/interactiveshell.py:3748: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


'My name is Alfred, and your name is Loki.'